# Machine Learning in Python - Project 

Due Friday, Apr 10th by 4 pm.

*Include contributors names here*

## Setup

*Install any packages here, define any functions if neeed, and load data*

In [ ]:
# Add any additional libraries or submodules below

# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler # scaling features
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold
from sklearn import metrics

# the warnings are too much for me
import warnings
warnings.filterwarnings('ignore')

# stats modules
from scipy.stats import chi2_contingency


# Constants 
DATA_FOLDER: str = "./data/"
UNICEF_DATA_CSV: str = DATA_FOLDER + "unicef_malawi.csv"
OUTPUT_FOLDER: str = "./output/"

In [6]:
# Load data  
df = pd.read_csv("unicef_malawi.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'unicef_malawi.csv'

# 1 Introduction

*This section should include a brief introduction to the task and the data (assume this is a report you are delivering to a professional body (e.g. UNICEF)).*

*Briefly outline the approaches being used and the conclusions that you are able to draw.*

## Introduction **Mark scheme**:
3 points (out of 30 for whole report).

An exceptional job at introducing the data and objectives, outlining and motivating the methods used, and summarizing the main conclusions. Can be passed
to the client with no editing required

Mental health conditions affect 8% of children 5-9 globally, and 15% of those aged 10-19 (Unicef, 2023). In lower and middle-income countries (LMICs) access to mental health resources are limited. The early-detection of mental health conditions is important in treatment, especially in children, where their brains are rapidly developing and early intervention can reduce progression of depression from childhood into adult life. UNICEF consulted with the DataBridge Foundation, a charity consisting of Edinburgh University Data Science experts, to build a binary classification model as to predict depression in children in LMICs. UNICEF operate under constrained resources, and it is crucial they allocate them where maximum utility and fairness is achieved. UNICEF’s provided us with their Multiple Indicator
Cluster Survey (MICS) data, consisting of information on the child, parent, and household, collected from asking questions via a series of telephone calls. In this report we present, key insights into the features in MICS that best predict childhood depression in that data; highlighting important signals of depression. Though causality analysis these signals naturally lead into our suggestions for prevention interventions.

In our exploratory data analysis we highlight the most important <span style="color:red">*X*</span> features. 
We prioritise model interpretability because our model will inform resource  allocation decisions aimed at reducing childhood depression in LMICs. Predictions must be explainable and defensible, supporting decisions that can be justified with clear logical argument. For this reason, we avoid deep neural networks in favour of inherently interpretable models. The primary limitation of this work is the data rather than the model. The data is subject to recall bias, social desirability bias, and straight-lining questions to finish them quickly.

# Exploratory Data Analysis and Feature Engineering

*Include a detailed discussion of the data with a particular emphasis on the features of the data that are relevant for the subsequent modeling. Including visualizations of the data is strongly encouraged - all code and plots must also be described in the write up. Think carefully about whether each plot needs to be included in your final draft and the appropriate type of plot and summary for each variable type - your report should include figures but they should be as focused and impactful as possible.*

*You should also split your data into training and testing sets, ideally before you look to much into the features and relationships with the target.*

*Additionally, this section should also motivate and describe any preprocessing / feature engineering of the data. Pipelines should be used and feature engineering steps that are be performed as part of an sklearn pipeline can be mentioned here but should be implemented in the following section.*

*All code and figures should be accompanied by text that provides an overview / context to what is being done or presented.*

## EDA **Mark scheme:**
6 points (out of 30 in entire report)

Exploratory Data Analysis and Feature Engineering: 6 points. Meaningful insights
into the data are carefully selected and presented in depth during the exploratory data analysis. Necessary preprocessing steps are carried out and described to a high standard. Feature
engineering steps are well motivated and fully described. Code is correct, interpretations are
sound, figures and summaries are appropriate for the data types, and can be passed to the
client with no editing required.

# 2 Exploratory Data Analysis and Feature Engineering

The data provided for this projcet in its raw form is varied and detailed yet flawed. With 87 features in the raw dataset the initals steps of the EDA will be focused around cutting down to a more manageable number, removing features which have no significant effect on the target. 

## 2.1 Rank correlations to target
We use Cramér's V to measure the association between each feature and the target, removing features with Cramér's V < 0.05, corresponding to less than 0.25% of variance explained individually. While individual features explain modest variance; reflecting the multifactorial and self-reported nature of the data, the combined predictive power of retained features is expected to be substantially higher, as the model learns from many weak signals jointly.

In [2]:
from scipy.stats import chi2_contingency
import numpy as np

def perform_cramers_v(
    *,
    df: pd.DataFrame,
    min_score: float = 0.05,
    target_column_name: str,
    ignore_columns: list,
) -> pd.DataFrame:
    """Performs cramers v test on the columns in the provided 'df' 
    against the 'target_column_name' column and returns a pd.DataFrame
    containing the signficiant features that meet the 'min_score'
    threshold. Ignores the 'ignore_columns'.
    """
    results = []
    df_without_target = df.drop(columns=ignore_columns)
    for col in df_without_target.columns:
        contingency_table = pd.crosstab(df[col], df[target_column_name])
        chi2, p, _, _ = chi2_contingency(contingency_table)
        n = contingency_table.sum().sum()
        min_dim = min(contingency_table.shape) - 1
        cramers_v = np.sqrt(chi2 / (n * min_dim))
        results.append({"feature_name": col, "p_value": p, "cramers_v": cramers_v})
    results_df = pd.DataFrame(results).sort_values("cramers_v", ascending=False)
    return results_df[results_df["cramers_v"] > min_score]

We remove w-score from our feature set as it represents a combined measure of wealth; arguably an alternative target variable rather than a predictor, and its inclusion would risk circular reasoning. Furt
hermore, although it could be an alternative target variable, it is unclear how it is actually obtained. Wealth indicators are factors which are stil likely to be considered in the final model however, it is preferable to avoid introudcing inexplicable elements into this report.

We additionally remove ethnicity to prevent the introduction of racial bias into a model that will inform resource allocation decisions. While ethnicity shows a weak but non-negligible association with depression (Cramér's V = 0.1, accounting for ~1% of variance), this association likely reflects proxy effects of socioeconomic status, cultural differences in self-reporting, or experiences of discrimination; factors better captured by other features in the dataset. We note that any future reintroduction of ethnicity would require significant ethical justification. This leaves us with 27 features.

In [3]:
def remove_features_from_df(
    *,
    df: pd.DataFrame,
    features_to_remove: list[str],
) -> pd.DataFrame:
    """Removes the features in 'features_to_remove' from the provided 'df' and returns the df.
    """
    return df[~df["feature_name"].isin(features_to_remove)] # removes features not in features_to_remove.

We now need to transform categories of features into numerical representations, either one-hot encodings, binary, or ordinal.

## 2.3 Full Pipeline

In [7]:
class FeatureNames:
    """Stores feature names used throughout.
    Does not containt all 87 features in the dataset.
    """
    CL3 = "CL3"
    CL13 = "CL13"
    FCD2H = "FCD2H"
    FCD2H_ANSWERED = "FCD2H_ANSWERED"
    FCD2J = "FCD2J"
    FCD2J_ANSWERED = "FCD2J_ANSWERED"
    LS1 = "LS1"
    LS2 = "LS2"
    LS3 = "LS3"
    LS4 = "LS4"
    FCF26 = "FCF26"
    FCF26_BINARY = "FCF26_binary"
    ETHNICITY = "ethnicity"
    WSCORE = "wscore"
    HH7 = "HH7"
    HC4 = "HC4"
    HC4_AGGREGATED = "HC4_AGGREGATED"
    HC5 = "HC5"
    HC5_AGGREGATED = "HC5_AGGREGATED"
    HC8 = "HC8"
    HC12 = "HC12"
    MA2 = "MA2"
    MA2_KNOWN = "MA2_KNOWN"
    MSTATUS = "MSTATUS"
    VT20 = "VT20"
    VT22A = "VT22A"
    VT22B = "VT22B"
    VT22C = "VT22C"
    VT22D = "VT22D"
    WS11 = "WS11"
    WS1 = "WS1"
    WS1_AGGREGATED = "WS1_AGGREGATED"
    WS4 = "WS4"
    WS7 = "WS7"

"""
====================
 feature_config map
====================
feature_config contains a map of feature name mapped to a dictionary of rules
as to how the feature will be transformed. "map" is used to map non-numeric
values to numeric ones. "numeric": True means the feature's values will be
coerced into numbers; i.e "30" will become 30.
"""
feature_config = {
    # LS1: To Mother: mother's happiness level, 4 categories
    FeatureNames.LS1: {
        "map": {
            "VERY UNHAPPY": 0,
            "SOMEWHAT UNHAPPY": 1,
            "NEITHER HAPPY NOR UNHAPPY": 2,
            "SOMEWHAT HAPPY": 3,
            "VERY HAPPY": 4,
            "NO RESPONSE": -1,
        },
    },
    # LS2 To Mother: mother's happiness level 0-10
    FeatureNames.LS2: {
        "map": {
            "NO RESPONSE": None,
        },
        "numeric": True,
    },
    # LS3: To Mother: Compared to this time last year, would you say that your
    #      life has improved, stayed more or less the same, or worsened,
    #      overall?
    FeatureNames.LS3: {
        "map": {
            "WORSENED": 0,
            "MORE OR LESS THE SAME": 1,
            "IMPROVED": 2,
            "NO RESPONSE": -1,
        },
    },
    # LS4: To Mother: And in one year from now, do you expect that your life
    #      will be better, will be more or less the same, or will be worse,
    #      overall?
    FeatureNames.LS4: {
        "map": {
            "WORSE": 0,
            "MORE OR LESS THE SAME": 1,
            "BETTER": 2,
            "NO RESPONSE": -1,
        },
    },
    # WS4: How long does it take for members of your household to go there,
    #      get water, and come back?
    #      "NO RESPONSE" implies they do not collect water and it is readily
    #      available, as WS4 question is only asked for certain water types
    #      involving travel.
    FeatureNames.WS4: {
        "map": {"DK": None, "MEMBERS DO NOT COLLECT": 0, "NO RESPONSE": 0, np.nan: 0}, "numeric": True,
    },
    # WS7: In the last month, has there been any time when your household did
    #      not have sufficient quantities of drinking water?
    FeatureNames.WS7: {
        "map": {
            "NO, ALWAYS SUFFICIENT": 0,
            "YES, AT LEAST ONCE": 1,
            "DK": -1,
            "NO RESPONSE": -1,
        },
    },
    # VT20: To Mother: How safe do you feel walking alone in your neighbourhood
    #       after dark?
    FeatureNames.VT20: {
        "map": {
            "VERY UNSAFE": 0,
            "UNSAFE": 1,
            "NEVER WALK ALONE AFTER DARK": 2,
            "SAFE": 3,
            "VERY SAFE": 4,
            "NO RESPONSE": -1,
        },
    },
    # VT22: In the past 12 months, have you personally felt discriminated
    #       against or harassed on the basis of the following grounds?
    # VT22A: A) Ethnic or immigration origin?
    FeatureNames.VT22A: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22B: B) Sex?
    FeatureNames.VT22B: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22C: C) Sexual orientation?
    FeatureNames.VT22C: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # VT22D: D) Age?
    FeatureNames.VT22D: {"map": {"NO": 0, "YES": 1, "DK": -1, "NO RESPONSE": -1}},
    # MA2: How old is your (husband/partner)?
    # -1 represents information not known, we'll later create a binary feature
    # indicating whether husband/partner's age is known in the dataset or not.
    FeatureNames.MA2: {"map": {"DK": -1, "NO RESPONSE": -1, np.nan: -1}, "numeric": True},
    # CL3: Since last (day of the week) about how many hours did (name) engage
    #      in (this activity/these activities), in total?
    #      Map np.nan values to 0 because if all answers to question CL2 are no,
    #      then question CL3 is skipped, implying 0 child labour hours worked.
    FeatureNames.CL3: {"map": {np.nan: 0}, "numeric": True},
    # CL13: Since last (day of the week), about how many hours did (name)
    #       engage in (this activity/these activities), in total?
    #       Again map np.nan values to 0. If all answers to question CL2 are
    #       no then question CL13 is skipped. There are 2 "NO RESPONSE" values.
    #       We map them to 0. Alternatively we could drop the two rows
    #       corresponding to "NO RESPONSE".
    FeatureNames.CL13: {
        "map": {
            "NO RESPONSE": 0,
            np.nan: 0
        },
        "numeric": True
    },
    # HC12: Does any member of your household have a mobile telephone?
    FeatureNames.HC12: {"map": {"NO": 0, "YES": 1, "NO RESPONSE": -1}},
    # MSTATUS: marital status of mother
    FeatureNames.MSTATUS: {
        "map": {
            "Never married/in union": 0,
            "Formerly married/in union": 1,
            "Currently married/in union": 2,
        },
    },
    #FCD: Question to child: Adults use certain ways to teach children the 
    # right behaviour or to address a behaviour problem. I will read various
    # methods that are used. Please tell me if you or any other adult in your
    # household has used this method with (name) in the past month.

    # FCD2H: Called (him/her) dumb, lazy or another name like that.
    #        We set "NO RESPONSE": -1, later we will introduce a new binary
    #        feature that indentifies whether the question was answered or not.
    FeatureNames.FCD2H: {"map": {"NO": 0, "YES": 1, "NO RESPONSE": -1, np.nan: -1}},
    # FCD2J: Hit or slapped (him/her) on the hand, arm, or leg.
    #        Ditto: We set "NO RESPONSE": -1, and later introduce binary feature
    #        identifying whether question was answered.
    FeatureNames.FCD2J: {"map": {"NO": 0, "YES": 1, "NO RESPONSE": -1, np.nan: -1}},
    # HC8: Does your household have electricity?
    FeatureNames.HC8: {
        "map": {
            "NO": 0,
            "YES, OFF-GRID (GENERATOR/ISOLATED SYSTEM)": 1,
            "YES, INTERCONNECTED GRID": 2,
            "NO RESPONSE": -1,
        },
    },
    
}

def apply_feature_config(*, df: pd.DataFrame, config: dict[str, dict]) -> pd.DataFrame:
    """Essentially encodes a pd.DataFrame's values according to a config dictionary.
    config dictionary is keyed by feature name. Values in the dictionary is a 
    map of rules to apply transformations to fields. "map" specifies how to map values.
    "numeric": True specifies to coerce the values into numeric values.

    """
    df = df.copy()
    for feature_name, rules in config.items():
        if feature_name not in df.columns:
            continue
        if value_map := rules.get("map"):
            # if there's a value map for the feature, then apply it.
            df[feature_name] = df[feature_name].map(value_map).fillna(df[feature_name])
        if rules.get("numeric"):
            # if numeric field is non-empty then coerce values to numeric.
            df[feature_name] = pd.to_numeric(df[feature_name], errors="coerce")       
    return df

In [ ]:
"""
 =========================================
  Feature Extraction and Engineering Main
 =========================================
  Section loads data, identifies the top features, manually removes features on further
  inspection. Then we convert all features to numerical representations, either binary,
  one-hot encodings (where a logical order does not make sense), and ordinal values in
  a logical order of significance. These steps are taken to transform the features
  into numerical representations that our model can "understand".
"""
    
# 1) Read unicef data into a Pandas DataFrame.
unicef_df = pd.read_csv(UNICEF_DATA_CSV)

# 2) Drop rows corresponding to nan values in "FCF26". We do this because we
#    will use 'FCF26' to derrive the target variable.
print(f"Dropping {unicef_df[FeatureNames.FCF26].isnull().mean() * 100:2f} % of rows corresponding to nan values in '{FeatureNames.FCF26}'.")
unicef_df = unicef_df.dropna(subset=[FeatureNames.FCF26])

# We introduce a depression_map that maps categories in the "FCF26" feature to
# binary values; where 0 = not depressed, and 1 = depressed.
depression_map = {
    "NEVER": 0, # Never sad implies never depressed (0).
    "A FEW TIMES A YEAR": 0, # Few times a year could be one-off sad life events. Therefore not depressed (0).
    "MONTHLY": 0, # Few times a year could be one-off monthly events. Therefore not depressed (0).
    "WEEKLY": 1, # Weekly is consistent and short frequent. Therefore depressed (1).
    "DAILY": 1, # Daily frequency of sadness suggests depression. Therefore depressed (1).
}
# 3) Remove rows corresponding to "NO RESPONSE" on "FCF26", the feature we will
#    use to derive the target. This Removes 0.78% of total rows.
unicef_df = unicef_df[unicef_df[FeatureNames.FCF26] != "NO RESPONSE"]

# 4) Map "FCF26" to binary feature of 0 = Not depressed, and 1 = Depressed
#    based on the depression_map.
unicef_df[FeatureNames.FCF26_BINARY] = unicef_df[FeatureNames.FCF26].map(depression_map)
print(f"Target: {FeatureNames.FCF26_BINARY} has {unicef_df[FeatureNames.FCF26_BINARY].isnull().mean() * 100:2f} % nan values")

# 5) Use Cramer's V score to select features that have a sufficient correlation
#    with the target "FCF26_binary".
#    Note correlation does not imply causation, so we cannot necessarily infer
#    causality without sufficient evidence of a mechanism that causally links
#    features highly correlated with the target.
significant_features_df = perform_cramers_v(
    df=unicef_df,
    # consider p > 0.05 a significant feature. This is tunable to be more/less
    # aggressive (higher being stricter).
    min_score=0.05,
    target_column_name=FeatureNames.FCF26_BINARY,
    ignore_columns=[
        FeatureNames.FCF26_BINARY, # FCF26_binary is target so remove FCF26_binary.
        FeatureNames.FCF26,  # target is derived from FCF26 so remove FCF26.
    ]
)

# 6) Remove some features manually on further inspection.
#     - Remove ethnicity feature to prevent racial bias. We need sufficient
#       ethical justification to include ethnicity.
#     - Remove wscore feature because it highly correlated with happiness.
#       Such that it could even be considered alternate target. When removing
#       wscore we note it as an important feature for our analysis.

significant_features_df = remove_features_from_df(
    df=significant_features_df,
    features_to_remove = [
        # Remove ethnicity feature to prevent racial bias.
        FeatureNames.ETHNICITY,
        FeatureNames.WSCORE,
    ]
)
significant_features: list[str] = significant_features_df["feature_name"].to_list()

# 7) Filter the unicef dataframe, including only significant features and the
#    target "FCF26_binary".
unicef_df_filtered = unicef_df[
    significant_features + [FeatureNames.FCF26_BINARY]
].copy()

# 8 For the HC categories, introduce new aggregate features that are the
# type of material with numerical ordering based on how developed the material
# types are.

# floor_categories maps floor categories to higher level categories.
# Ordered ascending from least developed to most developed.
floor_categories = {
    # NATURAL FLOOR: 0
    "EARTH / SAND": 0,
    "DUNG": 0,
    # RUDIMENTARY FLOOR: 1
    "WOOD PLANKS": 1,
    "PALM / BAMBOO": 1,
    # FINISHED FLOOR: 2
    "PARQUET OR POLISHED WOOD": 2,
    "VINYL OR ASPHALT STRIPS": 2,
    "CERAMIC TILES": 2,
    "CEMENT": 2,
    "CARPET": 2,

}
unicef_df_filtered[FeatureNames.HC4_AGGREGATED] = unicef_df_filtered[FeatureNames.HC4].map(floor_categories)

roof_categories = {
    # NO ROOF
    "NO ROOF": 0,
    # NATURAL ROOFING
    "THATCH / PALM LEAF": 1,
    "SOD": 1,
    # RUDIMENTARY ROOFING
    "RUSTIC MAT": 2,
    "PALM / BAMBOO": 2,
    "WOOD PLANKS": 2,
    "CARDBOARD": 2,
    # FINISHED ROOFING
    "IRON SHEETS / METAL / TIN": 3,
    "WOOD": 3,
    "CALAMINE / CEMENT FIBRE": 3,
    "CERAMIC TILES": 3,
    "CEMENT": 3,
    "ROOFING SHINGLES": 3,
}
unicef_df_filtered[FeatureNames.HC5_AGGREGATED] = unicef_df_filtered[FeatureNames.HC5].map(roof_categories)

drinking_water_types = {
    # UN-PIPED WATER
    "TUBE WELL / BOREHOLE": 0,
    "DUG WELL: PROTECTED WELL": 0,
    "DUG WELL: UNPROTECTED WELL": 0,
    "SPRING: PROTECTED SPRING": 0,
    "SPRING: UNPROTECTED SPRING": 0,
    "RAINWATER": 0,
    "TANKER-TRUCK": 0,
    "CART WITH SMALL TANK ": 0,
    "WATER KIOSK": 0,
    "SURFACE WATER (RIVER, DAM, LAKE, POND, STREAM, CANAL, IRRIGATION CHANNEL)": 0,
    

    # PACKAGED WATER
    "PACKAGED WATER: BOTTLED WATER": 1,
    "PACKAGED WATER: SACHET WATER": 1,

    # PIPED WATER
    "PIPED WATER: PIPED INTO DWELLING": 2,
    "PIPED WATER: PIPED TO YARD / PLOT": 2,
    "PIPED WATER: PIPED TO NEIGHBOUR": 2,
    "PIPED WATER: PUBLIC TAP / STANDPIPE": 2,
}
unicef_df_filtered[FeatureNames.WS1_AGGREGATED] = unicef_df_filtered[FeatureNames.WS1].map(drinking_water_types)

# 8) Apply feature config to transform features into a numerical format that
#    our model can understand. Nominal Features we one-hot encode, and drop the
#    first column in the encoding to remove redundant information.
nominal_features = [
    # HC4: Main material of the dwelling floor.
    FeatureNames.HC4,
    # HC5: Main material of the roof.
    FeatureNames.HC5,
    # HH7: Region
    FeatureNames.HH7,
    # WS1: What is the main source of drinking water used by members of your
    # household?
    FeatureNames.WS1,
    # WS11: What kind of toilet facility do members of your household usually
    # use?
    FeatureNames.WS11,
]
unicef_df_filtered = pd.get_dummies(
    unicef_df_filtered,
    columns=nominal_features,
    # We set drop_first=True to reduce redundant information, a matrix of all
    # zeroes implies the first category.
    drop_first=True,
)

# 9) For the categorical features where a natural ordering makes sense, we
#    transform them using a map, mapping the category to a numeric value our
#    model will understand.
unicef_df_filtered = apply_feature_config(df=unicef_df_filtered, config=feature_config)

# 10) Introduce additional features: Whether Violence Questions were answered
#     or not. 1 = Yes, 0 = No.
unicef_df_filtered[FeatureNames.FCD2H_ANSWERED] = (unicef_df_filtered[FeatureNames.FCD2H]  != -1).astype(int)
unicef_df_filtered[FeatureNames.FCD2J_ANSWERED] = (unicef_df_filtered[FeatureNames.FCD2J]  != -1).astype(int)

# And whether husband/partner age is known or not in the dataset.
unicef_df_filtered[FeatureNames.MA2_KNOWN] = (unicef_df_filtered[FeatureNames.MA2]  != -1).astype(int)

# Drop the 1 row where MSTATUS is "9.0"; because the value "9.0" does not make sense.
unicef_df_filtered = unicef_df_filtered[unicef_df_filtered["MSTATUS"] != "9.0"]

null_summary = pd.DataFrame({
    "count": unicef_df_filtered.isnull().sum(),
    "% null": unicef_df_filtered.isnull().mean() * 100
})

# 11) Identify null values: After the feature engineering steps, we have some null values, all less than 2.63%.
print(null_summary[null_summary["count"] > 0].round(2))

# 12) Fill null values with mode: Proportion of null values is small enough to fill in with mode.
for col in unicef_df_filtered.columns:
    unicef_df_filtered[col] = unicef_df_filtered[col].fillna(unicef_df_filtered[col].mode()[0])

# 3 Model Fitting and Tuning

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. As with figures, any included code must also be addressed in the text of the document.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

For the purposes of this classification, we considered classification trees, bagging, random forests, support vector machines, and logisitic regression. These frameworks were implemented with adequate tuning to produce a predictive model. The support vector machine approach failed to fit in a reasonable amount of runtime, as after over 45 minutes the pipeline still failed to produce results. For this reason, the support vector machine approach was removed from consideration for being impractical for the application and audience at hand. Metrics from the remaining approaches are summarized in the table below.

Of these metrics, the most pertinent is recall, or the rate at which the model correctly identifies children exhibiting depression as being depressed. In machine learning terms, this is the number of true positives (children with depression predicted to be depressed) divided by the sum of true positives and false negatives (children with depression predicted not to be depressed). We choose this metric as most important because failure to identify depression in children with depression, i.e. false negatives, lead to the most significant negative public health outcomes. 

In this case, the model with the best recall is the logistic regression model. This follows from the fact that logistic regression can be tuned specifically to perform better according to specific performance metrics, and thus could be tuned specifically to maximise recall. Logistic regression was therefore chosen as the approach from which to further develop a predictive model. 

## Model Fitting and Tuning **Mark Scheme**:

6 points (out of 30 for report). An effective model is selected that is suitable for
the problem at hand, and a compelling case for that model is made. The model is properly
fit and tuned. A sound comparison with a baseline model is provided. Code is correct, model
is well justified, and can be passed to the client with no editing required.

In [ ]:
def get_coefs(m, plot = False, feature_names = None, figsize = (5,5), figtitle = None, intercept = True):
    """Returns model coefficients in a data frame for a fitted linear model.
    
    Args:
        m: sklearn linear model object or pipeline with linear model as final step
        plot: boolean value, should coefficients be plotted with error bars
        feature_names: list of feature names to use in the plot 
        figsize: tuple defining figure size
        figtitle: string defining figure title
        intercept: boolean value, should intercept be included in the plot
    """
    
    # Extract intercept and coefficients into a single array
    w0 = m[-1].intercept_ if isinstance(m, sklearn.pipeline.Pipeline) else m.intercept_
    w1 = m[-1].coef_ if isinstance(m, sklearn.pipeline.Pipeline) else m.coef_
    w = np.concatenate((w0.reshape(-1), w1.reshape(-1)))
    # Extract name of features
    if feature_names is None:
        feature_names = m[:-1].get_feature_names_out() if isinstance(m, sklearn.pipeline.Pipeline) else m.feature_names_in_
    feature_names = np.concatenate((['intercept'], feature_names))
    # Create a data frame
    w_df = pd.DataFrame({'feature': feature_names, 'coef': w}).sort_values ("coef", ascending=False)

    if plot:
        if not intercept:
            w_df = w_df[w_df['feature'] != 'intercept']
        plt.figure(figsize=figsize)
        plt.barh(w_df['feature'], w_df['coef'])
        plt.ylabel('Features')
        plt.xlabel('Coefficient Value')
        plt.axvline(x=0, color=".5")
        if figtitle is not None:
            plt.title(figtitle)
        plt.grid()
        plt.show()
    
    return  w_df

# Pipeline 
log_pipe_l2 = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight={0:1, 1:7.86}, random_state=42, penalty='l2', solver='liblinear',
        max_iter=1000)
)

# Possible C values: 
C_list = np.linspace(-5,5, num=100)

# Grid search CV:
log_rs = GridSearchCV(log_pipe_l2, 
                      param_grid={'logisticregression__C': C_list},
                      scoring = ["accuracy", "f1","recall","precision"], #Evaluation metrics to compute on validation sets
                      cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                      refit = "recall", # Refits the best model on the entire dataset using the accuracy metric 
                      return_train_score = True)

# Tune the model with grid search:
log_rs.fit(X_train, y_train)

# Extract only mean and split scores
cv_accuracy = pd.DataFrame(
    data = log_rs.cv_results_
).filter(
    # Extract the split#_test_f1 and mean_test_f1 columns
    regex = '(split[0-4]+|mean)_test_recall'
).assign(
    # Add the alphas as a column
    C = C_list
)
# Reshape the data frame for plotting
d = cv_accuracy.melt(
    id_vars=('C','mean_test_recall'),
    var_name='fold',
    value_name='recall'
)
# Plot the validation scores across folds
plt.figure(figsize=(7,5))
sns.lineplot(x='C', y='recall', color='black', errorbar=None, data = d)  # Plot the mean score in black.
sns.lineplot(x='C', y='recall', hue='fold', data = d) # Plot the curves for each fold in different colors
plt.show()

log_pipe_l2 = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight={0:1, 1:7.86}, random_state=42, penalty='l2', solver='liblinear',
        max_iter=1000, C=.75)
)

log_pipe_l2.fit(X_train, y_train)
get_coefs(log_pipe_l2, plot=True)
y_pred = log_pipe_l2.predict(X_test)
ConfusionMatrixDisplay.from_estimator(log_pipe_l2, X_test, y_test)
plt.show()
print(f"Accuracy: {metrics.accuracy_score(y_test, y_pred)}")
print(f"Precision: {metrics.precision_score(y_test, y_pred)}")
print(f"Sensitivity/Recall: {metrics.recall_score(y_test, y_pred)}")
print(f"Specificity: {metrics.recall_score(y_test, y_pred, pos_label=0.0)}")
print(f"F Score: {metrics.f1_score(y_test, y_pred)}")

log_pipe_bal = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", random_state=42, penalty='l2', solver='liblinear',
        max_iter=1000)
)

# Grid search CV:
log_rs = GridSearchCV(log_pipe_bal, 
                      param_grid={'logisticregression__C': C_list},
                      scoring = ["accuracy", "f1","recall","precision"], #Evaluation metrics to compute on validation sets
                      cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                      refit = "recall", # Refits the best model on the entire dataset using the accuracy metric 
                      return_train_score = True)

# Tune the model with grid search:
log_rs.fit(X_train, y_train)

# Extract only mean and split scores
cv_accuracy = pd.DataFrame(
    data = log_rs.cv_results_
).filter(
    # Extract the split#_test_f1 and mean_test_f1 columns
    regex = '(split[0-4]+|mean)_test_recall'
).assign(
    # Add the alphas as a column
    C = C_list
)
# Reshape the data frame for plotting
d = cv_accuracy.melt(
    id_vars=('C','mean_test_recall'),
    var_name='fold',
    value_name='recall'
)
# Plot the validation scores across folds
plt.figure(figsize=(7,5))
sns.lineplot(x='C', y='recall', color='black', errorbar=None, data = d)  # Plot the mean score in black.
sns.lineplot(x='C', y='recall', hue='fold', data = d) # Plot the curves for each fold in different colors
plt.show()

log_pipe_bal = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", random_state=42, penalty='l2', solver='liblinear',
        max_iter=1000, C=1.5)
)

log_pipe_bal.fit(X_train, y_train)
get_coefs(log_pipe_bal, plot=True)
y_pred = log_pipe_bal.predict(X_test)
ConfusionMatrixDisplay.from_estimator(log_pipe_bal, X_test, y_test)
plt.show()

print(f"Accuracy: {metrics.accuracy_score(y_test, y_pred)}")
print(f"Precision: {metrics.precision_score(y_test, y_pred)}")
print(f"Sensitivity/Recall: {metrics.recall_score(y_test, y_pred)}")
print(f"Specificity: {metrics.recall_score(y_test, y_pred, pos_label=0.0)}")
print(f"F Score: {metrics.f1_score(y_test, y_pred)}")

# 4 Interpretation and Discussion

*In this section you should provide a general overview of your final model, its performance, and reliability. You should discuss what the implications of your model are in terms of the included features, estimated parameters and relationships, predictive performance, and anything else you think is relevant.*

*This should be written with a target audience of a government official, who understands the issues associated with mental health but may only have university level mathematics (not necessarily postgraduate statistics or machine learning). Your goal should be to highlight to this audience how your model can useful. You should also discuss potential limitations or directions of future improvement of your model.*

*Finally, you should include recommendations on factors that may increase the risk of depression, which may be useful for the government officials and health care workers to improve their understanding of the condition, and potentially assit in the development of effective social and health policies and interventions.*

*Keep in mind that a negative result, i.e. a model that does not work well predictively, that is well explained and justified in terms of why it failed will likely receive higher marks than a model with strong predictive performance but with poor or incorrect explanations / justifications.*

## Intrepretation, Discussion and Conclusion **Mark Scheme**:

6 points. The interpretation and discussion of the model fit results is exemplary and provides useful insight. The discussion and
results are not overstated, and possible limitations of the model and data are included. Recommendations are presented for the client, and important questions for future research in the
field are identified. Interpretations are sound, and can be passed to the client with no editing
required.

## Remaining marks **Mark Scheme:**

Overall Writing: 3 points. The writing is rigorous, yet flows naturally; there are no lapses
in phrasing, grammar, or spelling, resulting in a report that is enjoyable to read throughout.

Overall Coding: 3 points. Code is well-structured and easy to follow and it solves the
problem as intended. Code has appropriate comments and documentation and it uses best
practices and industry standards.

Overall Visualizations: 3 points. Clear, well-organized figures/tables with appropriate
labels and integration with project and proper referencing/citation. The report evidences
exceptional attention to detail in the presentation of figures and tables.


# 5 Conclusion

# Generative AI statement

*Include a statement on how generative AI was used in the project and report.*

# References

<ol>
    <li>UNICEF (2024) Global Annual Results Report 2023: Mental health. UNICEF. Available at: https://www.unicef.org/reports/global-annual-results-report-2023-mental-health
 (Accessed: 14 April 2026)</li>
</ol>

# Appendix

## A) Plots of all features

![alt text](output/appendix_all_plots.png)

In [ ]:
# Run the following to render to PDF
!jupyter nbconvert --to pdf project.ipynb